# the below code contain informatoon about data cleaning 

## the code contains the following:
1. import any type of data including text data
2. cleaning data  

In [ ]:
def manual_parse_text_file(path):
    try:
        with open(path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        df = pd.DataFrame(data)
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    return None

In [ ]:
# ============================================================
# GEOCHEMICAL DATA ANALYSIS - JUPYTER VERSION
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# PART 1: DATA CLEANING ENGINE
# ============================================================

def clean_any_data(path, sheet_name=None, **kwargs):
    """Universal data cleaner for Jupyter notebooks"""
    print(f"\n📁 READING FILE: {os.path.basename(path)}")
    print("=" * 60)
    
    file_ext = Path(path).suffix.lower()
    df = None
    encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1', 'utf-16', 'cp1250']
    
    if file_ext in ['.xlsx', '.xls', '.xlsm']:
        df = read_excel_file(path, sheet_name)
    elif file_ext in ['.csv', '.txt', '.dat', '.tsv']:
        df = read_text_file(path, encodings, **kwargs)
        if df is None or df.empty:
            df = manual_parse_text_file(path)
    else:
        df = manual_parse_text_file(path)
    
    if df is None or df.empty:
        print("❌ Could not read file")
        return None
    
    print(f"📊 Raw shape: {df.shape}")
    df = clean_dataframe(df)
    print(f"✅ Final shape: {df.shape}")
    return df

def read_excel_file(path, sheet_name):
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine='openpyxl')
        print("✅ Excel file loaded")
        return df
    except:
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, engine='xlrd')
            print("✅ Excel file loaded (xlrd)")
            return df
        except Exception as e:
            print(f"⚠️ Excel read failed: {e}")
            return None

def read_text_file(path, encodings, **kwargs):
    delimiters = [';', ',', '\t', '|', ' ']
    for encoding in encodings:
        for delim in delimiters:
            try:
                df = pd.read_csv(path, sep=delim, encoding=encoding, engine='python', on_bad_lines='skip', **kwargs)
                if df is not None and not df.empty and df.shape[1] > 1:
                    print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                    return df
            except:
                continue
    return None

def manual_parse_text_file(path):
    try:
        with open(path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        df = pd.DataFrame(data)
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    return None

def clean_dataframe(df):
    if df is None or df.empty:
        return df
    df = split_single_column(df)
    df = clean_column_names(df)
    df = remove_header_rows(df)
    df = standardize_missing_values(df)
    df = clean_numeric_columns(df)
    df = clean_text_columns(df)
    df = remove_empty(df)
    return df

def split_single_column(df):
    if df is None or df.empty or df.shape[1] != 1:
        return df
    col_name = df.columns[0]
    col_data = df[col_name].astype(str)
    sample = col_data.iloc[0] if len(col_data) > 0 else ''
    delimiters = [';', ',', '\t', '|']
    for delim in delimiters:
        if delim in sample:
            split_data = col_data.str.split(delim, expand=True)
            try:
                pd.to_numeric(split_data.iloc[0, 0])
                split_data.columns = [f'col_{i+1}' for i in range(split_data.shape[1])]
            except:
                headers = split_data.iloc[0].tolist()
                split_data = split_data.iloc[1:].copy()
                headers = [str(h).strip() for h in headers]
                split_data.columns = headers
            return split_data.reset_index(drop=True)
    return df

def clean_column_names(df):
    if df is None or df.empty:
        return df
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.loc[:, ~df.columns.astype(str).str.contains(r',+$', na=False)]
    df.columns = [f'col_{i}' if str(col).strip() == '' or pd.isna(col) else col for i, col in enumerate(df.columns)]
    df.columns = [re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col)).strip('_') for col in df.columns]
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

def remove_header_rows(df):
    if df is None or df.empty or len(df) == 0:
        return df
    first_row = df.iloc[0].astype(str)
    header_keywords = ['sample', 'point', 'id', 'name', 'date', 'time', 'value', 'measurement', 'parameter', 'unit', 'ppm']
    header_score = sum(1 for val in first_row if any(keyword in str(val).lower() for keyword in header_keywords))
    if len(first_row) > 0 and header_score / len(first_row) > 0.3:
        df = df.iloc[1:].reset_index(drop=True)
    return df

def standardize_missing_values(df):
    if df is None or df.empty:
        return df
    missing_values = ['', 'nan', 'NaN', 'NAN', 'null', 'NULL', 'None', 'na', 'NA', '-', '--', '---', 'n/a', 'N/A', '?']
    df = df.replace(missing_values, np.nan)
    for col in df.columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'^<.*$', 'nan', regex=True)
            df[col] = df[col].astype(str).str.replace(r'^>.*$', 'nan', regex=True)
    return df

def clean_numeric_columns(df):
    if df is None or df.empty:
        return df
    text_cols = ['sample', 'point_id', 'point', 'id', 'name', 'lithology', 'description', 'comment', 'remarks', 'unit', 'method', 'date', 'time']
    def clean_numeric(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().replace(',', '.')
        if x.startswith('<') or x.startswith('>'):
            return np.nan
        if x in ['', 'nan', 'NaN', 'NAN', 'null', 'NULL']:
            return np.nan
        x = re.sub(r'[^\d.\-eE+]', '', x)
        try:
            return float(x)
        except:
            return np.nan
    for col in df.columns:
        col_lower = col.lower()
        is_text = any(text in col_lower for text in text_cols)
        if not is_text:
            try:
                df[col] = df[col].map(clean_numeric)
            except:
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
    return df

def clean_text_columns(df):
    if df is None or df.empty:
        return df
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('', np.nan)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
    return df

def remove_empty(df):
    if df is None or df.empty:
        return df
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    return df

# ============================================================
# PART 2: STATISTICAL ANALYSIS ENGINE
# ============================================================

class StatisticalAnalyzer:
    """Comprehensive statistical analysis for geochemical data"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        self.run_all_analyses()
    
    def run_all_analyses(self):
        """Run all statistical analyses"""
        self.calculate_variance()
        self.calculate_covariance()
        self.calculate_correlation()
        self.calculate_eigen()
        self.perform_pca()
        self.standardize_data()
        self.exploratory_analysis()
    
    def calculate_variance(self):
        """Calculate variance for all numeric columns"""
        self.results['variance'] = self.numeric_df.var()
        self.results['variance_dict'] = self.numeric_df.var().to_dict()
        self.results['std'] = self.numeric_df.std()
        self.results['mean'] = self.numeric_df.mean()
        self.results['median'] = self.numeric_df.median()
    
    def calculate_covariance(self):
        """Calculate covariance matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['covariance_matrix'] = clean_data.cov()
        else:
            self.results['covariance_matrix'] = None
    
    def calculate_correlation(self):
        """Calculate correlation matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['correlation_matrix'] = clean_data.corr()
        else:
            self.results['correlation_matrix'] = None
    
    def calculate_eigen(self):
        """Calculate eigenvalues and eigenvectors"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2:
            self.results['eigenvalues'] = None
            self.results['eigenvectors'] = None
            return
        
        try:
            cov = clean_data.cov().values
            eigenvals, eigenvecs = np.linalg.eig(cov)
            
            # Sort by eigenvalues (largest first)
            idx = np.argsort(eigenvals)[::-1]
            eigenvals = eigenvals[idx]
            eigenvecs = eigenvecs[:, idx]
            
            self.results['eigenvalues'] = eigenvals
            self.results['eigenvectors'] = eigenvecs
            self.results['explained_variance'] = eigenvals / eigenvals.sum() * 100
        except Exception as e:
            self.results['eigenvalues'] = None
            self.results['eigenvectors'] = None
            print(f"⚠️ Eigen decomposition failed: {e}")
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2 or self.results['eigenvectors'] is None:
            self.results['pca_scores'] = None
            self.results['pca_loadings'] = None
            return
        
        try:
            centered = clean_data - clean_data.mean()
            pca_scores = centered @ self.results['eigenvectors']
            self.results['pca_scores'] = pca_scores
            self.results['pca_loadings'] = self.results['eigenvectors']
            self.results['pca_columns'] = [f'PC{i+1}' for i in range(pca_scores.shape[1])]
            self.results['pca_df'] = pd.DataFrame(pca_scores, columns=self.results['pca_columns'])
        except Exception as e:
            print(f"⚠️ PCA failed: {e}")
    
    def standardize_data(self):
        """Standardize data (Z-score normalization)"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 0:
            self.results['standardized'] = (clean_data - clean_data.mean()) / clean_data.std()
            self.results['min_max_scaled'] = (clean_data - clean_data.min()) / (clean_data.max() - clean_data.min())
    
    def exploratory_analysis(self):
        """Full exploratory data analysis"""
        self.results['eda'] = {}
        self.results['eda']['shape'] = self.df.shape
        self.results['eda']['numeric_shape'] = self.numeric_df.shape
        self.results['eda']['missing'] = self.df.isnull().sum()
        self.results['eda']['missing_pct'] = self.df.isnull().sum() / len(self.df) * 100
        self.results['eda']['skewness'] = self.numeric_df.skew()
        self.results['eda']['kurtosis'] = self.numeric_df.kurtosis()
        self.results['eda']['quantiles'] = self.numeric_df.quantile([0.25, 0.5, 0.75])
        
        if len(self.numeric_df.columns) > 1:
            corr = self.numeric_df.corr()
            self.results['eda']['correlation_summary'] = {
                'max_corr': corr.max().max(),
                'min_corr': corr.min().min(),
                'high_corr_pairs': [(col1, col2, corr.loc[col1, col2]) 
                                   for i, col1 in enumerate(corr.columns)
                                   for col2 in corr.columns[i+1:]
                                   if abs(corr.loc[col1, col2]) > 0.7]
            }
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'variance': self.results.get('variance_dict', {}),
            'mean': self.results.get('mean', {}),
            'std': self.results.get('std', {}),
            'correlation_matrix': self.results.get('correlation_matrix', None),
            'eigenvalues': self.results.get('eigenvalues', None),
            'eigenvectors': self.results.get('eigenvectors', None),
            'explained_variance': self.results.get('explained_variance', None),
            'pca_scores': self.results.get('pca_df', None),
            'missing_values': self.results.get('eda', {}).get('missing', None),
            'skewness': self.results.get('eda', {}).get('skewness', None)
        }
    
    def get_pca_report(self, n_components=2):
        """Get a detailed PCA report"""
        if self.results['pca_scores'] is None:
            return "PCA could not be performed."
        
        report = []
        report.append("="*70)
        report.append("PRINCIPAL COMPONENT ANALYSIS REPORT")
        report.append("="*70)
        report.append(f"\nSamples: {self.results['pca_scores'].shape[0]}")
        report.append(f"Variables: {self.results['pca_scores'].shape[1]}")
        report.append("\nEigenvalues and Explained Variance:")
        report.append("-"*40)
        
        for i in range(len(self.results['eigenvalues'])):
            ev = self.results['eigenvalues'][i]
            ev_pct = self.results['explained_variance'][i]
            cum_pct = self.results['explained_variance'][:i+1].sum()
            report.append(f"  PC{i+1}: λ = {ev:.4f}  ({ev_pct:.1f}%  cumulative: {cum_pct:.1f}%)")
        
        report.append("\nPrincipal Component Loadings (first 3):")
        report.append("-"*40)
        variables = self.numeric_df.dropna().columns.tolist()
        for i in range(min(3, self.results['eigenvectors'].shape[1])):
            report.append(f"  PC{i+1}:")
            for j, var in enumerate(variables):
                loading = self.results['eigenvectors'][j, i]
                report.append(f"    {var}: {loading:.4f}")
        
        return "\n".join(report)
    
    def get_eda_report(self):
        """Get a full EDA report"""
        if self.results['eda'] is None:
            return "EDA could not be performed."
        
        report = []
        report.append("="*70)
        report.append("EXPLORATORY DATA ANALYSIS REPORT")
        report.append("="*70)
        report.append(f"\nShape: {self.results['eda']['shape'][0]} rows × {self.results['eda']['shape'][1]} columns")
        report.append(f"Numeric Columns: {self.results['eda']['numeric_shape'][1]}")
        
        report.append("\nMissing Values:")
        report.append("-"*40)
        missing = self.results['eda']['missing']
        missing_pct = self.results['eda']['missing_pct']
        for col in missing.index:
            if missing[col] > 0:
                report.append(f"  {col}: {missing[col]} ({missing_pct[col]:.1f}%)")
        
        report.append("\nSkewness:")
        report.append("-"*40)
        skewness = self.results['eda']['skewness']
        for col, val in skewness.items():
            if not pd.isna(val):
                shape = "highly skewed" if abs(val) > 1 else "moderately skewed" if abs(val) > 0.5 else "symmetric"
                report.append(f"  {col}: {val:.4f} ({shape})")
        
        report.append("\nKurtosis:")
        report.append("-"*40)
        kurtosis = self.results['eda']['kurtosis']
        for col, val in kurtosis.items():
            if not pd.isna(val):
                shape = "heavy tails" if val > 3 else "light tails"
                report.append(f"  {col}: {val:.4f} ({shape})")
        
        return "\n".join(report)

# ============================================================
# PART 3: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation_matrix(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("Need at least 2 numeric columns for correlation plot")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, fmt='.2f')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(analyzer, figsize=(10, 6)):
    """Plot PCA scores"""
    if analyzer.results['pca_scores'] is None:
        print("No PCA results available")
        return
    
    pca_df = analyzer.results['pca_df']
    explained_var = analyzer.results['explained_variance']
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Scatter plot of PC1 vs PC2
    ax1 = axes[0]
    ax1.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.7, c='steelblue', edgecolors='white', linewidth=0.5)
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel(f'PC1 ({explained_var[0]:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained_var[1]:.1f}%)')
    ax1.set_title('PCA Scores')
    ax1.grid(True, alpha=0.3)
    
    # Explained variance bar plot
    ax2 = axes[1]
    components = [f'PC{i+1}' for i in range(len(explained_var))]
    ax2.bar(components, explained_var, color='steelblue', edgecolor='white')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_feature_distributions(df, cols=None, figsize=(12, 8)):
    """Plot histograms for numeric columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:6]  # Limit to first 6 columns
    
    n_cols = min(len(cols), 6)
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten() if n_rows > 1 else [axes]
    
    for i, col in enumerate(cols):
        if i < len(axes) and col in numeric_df.columns:
            axes[i].hist(numeric_df[col].dropna(), bins=20, edgecolor='black', alpha=0.7, color='steelblue')
            axes[i].set_title(col, fontsize=10)
            axes[i].set_xlabel('Value')
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)
    
    # Remove empty subplots
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

def plot_scatter_matrix(df, cols=None, figsize=(12, 10)):
    """Plot scatter matrix for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    selected_data = numeric_df[cols].dropna()
    if len(selected_data) < 2:
        print("Not enough data for scatter matrix")
        return
    
    pd.plotting.scatter_matrix(selected_data, figsize=figsize, alpha=0.5, 
                                diagonal='hist', color='steelblue')
    plt.suptitle('Scatter Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 4: COMPLETE WORKFLOW IN JUPYTER
# ============================================================

def load_and_analyze(file_path, sheet_name=None):
    """
    Complete workflow: load, clean, and analyze geochemical data
    """
    # Load and clean
    df = clean_any_data(file_path, sheet_name)
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    print("\n" + "="*60)
    print("DATASET SUMMARY")
    print("="*60)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Numeric columns: {len(df.select_dtypes(include=[np.number]).columns)}")
    print("\nFirst 5 rows:")
    print(df.head())
    
    # Analyze
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS")
    print("="*60)
    
    analyzer = StatisticalAnalyzer(df)
    
    # Print reports
    print("\n" + analyzer.get_eda_report())
    print("\n" + analyzer.get_pca_report())
    
    # Display visualizations
    print("\n" + "="*60)
    print("VISUALIZATIONS")
    print("="*60)
    
    # Plot correlation matrix
    plot_correlation_matrix(df, title="Geochemical Correlation Matrix")
    
    # Plot PCA scores
    plot_pca_scores(analyzer)
    
    # Plot feature distributions
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        plot_feature_distributions(df, cols=numeric_cols[:6])
    
    return df, analyzer

# ============================================================
# EXAMPLE USAGE IN JUPYTER
# ============================================================

# Example 1: Load and analyze a file
# df, analyzer = load_and_analyze('your_data.csv')

# Example 2: Load data only
# df = clean_any_data('your_data.csv')

# Example 3: Analyze existing dataframe
# analyzer = StatisticalAnalyzer(df)

# Example 4: Get specific results
# summary = analyzer.get_summary()
# print("Variance:", summary['variance'])
# print("Correlation Matrix:", summary['correlation_matrix'])
# print("PCA Scores:", summary['pca_scores'])

# Example 5: Export cleaned data
# df.to_csv('cleaned_data.csv', sep=';', index=False, encoding='utf-8-sig')

# Example 6: Save PCA results
# if analyzer.results['pca_df'] is not None:
#     analyzer.results['pca_df'].to_csv('pca_scores.csv', sep=';', index=False)

print("\n" + "="*60)
print("✅ GEOCHEMICAL DATA ANALYSIS TOOLKIT LOADED")
print("="*60)
print("\nAvailable functions:")
print("  • clean_any_data(path) - Load and clean data")
print("  • StatisticalAnalyzer(df) - Full statistical analysis")
print("  • plot_correlation_matrix(df) - Correlation heatmap")
print("  • plot_pca_scores(analyzer) - PCA visualization")
print("  • plot_feature_distributions(df) - Histograms")
print("  • plot_scatter_matrix(df) - Scatter matrix")
print("\nExample workflow:")
print("  df = clean_any_data('your_file.csv')")
print("  analyzer = StatisticalAnalyzer(df)")
print("  plot_correlation_matrix(df)")
print("  plot_pca_scores(analyzer)")